# Utah FORGE 00 Load

Inspect raw files for the Utah FORGE pipeline and save an explicit dataset summary under `results/utah_forge/`.


In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import get_dataset_config
from src.datasets.utah_forge import build_utah_forge_summary
from src.io.utah_forge import load_utah_forge_dataset
from src.utils.plotting import plot_signal_panel

CONFIG = get_dataset_config("utah_forge")
RESULTS_DIR = CONFIG.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
summary = build_utah_forge_summary()
summary_path = RESULTS_DIR / "dataset_summary.json"
inventory_path = RESULTS_DIR / "raw_file_inventory.json"
readme_path = RESULTS_DIR / "README.md"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
readme_path.write_text(
    "\n".join(
        [
            "# Utah FORGE Dataset Summary",
            "",
            f"- Source URL: {summary['dataset']['source_url']}",
            f"- Analysis ready: {summary.get('analysis_ready', False)}",
            f"- Notes: {summary.get('notes', summary['dataset'].get('notes', ''))}",
            f"- Raw file: {summary.get('raw_file')}",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
inventory = {
    "dataset": "utah_forge",
    "raw_dir": str(CONFIG.raw_dir),
    "filenames_found": sorted(path.name for path in CONFIG.raw_dir.iterdir() if path.is_file()),
    "file_types_found": sorted({path.suffix.lower() or "<no_extension>" for path in CONFIG.raw_dir.iterdir() if path.is_file()}),
    "loader": "load_utah_forge_dataset",
    "loader_status": "ok" if summary.get("raw_file") and not summary.get("error") else "error",
    "loader_raw_file": summary.get("raw_file"),
    "parsed_columns": summary.get("schema", {}).get("columns", []),
    "available_variables": summary.get("available_variables", summary.get("schema", {}).get("columns", [])),
    "column_mapping": summary.get("column_mapping", {}),
}
inventory_path.write_text(json.dumps(inventory, indent=2), encoding="utf-8")
if summary.get("raw_file"):
    raw_df, _ = load_utah_forge_dataset()
    print(raw_df.head())


{
  "dataset": {
    "name": "utah_forge",
    "label": "Utah FORGE Laboratory Shear Experiments",
    "source_url": "https://catalog.data.gov/dataset/utah-forge-laboratory-shear-experiments-linking-fault-roughness-friction-permeability-and-",
    "raw_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\data\\utah_forge",
    "results_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\results\\utah_forge",
    "notebooks_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\notebooks\\utah_forge",
    "preferred_raw_names": [
      "0_README.txt"
    ],
    "raw_globs": [
      "*_datatable.mat",
      "*.mat",
      "*.txt"
    ],
    "column_aliases": {
      "time": [
        "time",
        "Time",
        "t",
        "seconds"
      ],
      "tau": [
        "tau",
        "shear stress",
        "shear_stress"
      ],
      "displacement": [
        "d_int",
        "displacement",
        "slip",
        "u",
        "shear_displacement",
        "d_ext"
      ],
      "v

In [3]:
if summary.get("raw_file"):
    numeric_cols = raw_df.select_dtypes(include=["number"]).columns.tolist()
    if summary.get("column_mapping", {}).get("time") and summary.get("column_mapping", {}).get("tau"):
        preview_df = raw_df[[summary["column_mapping"]["time"], summary["column_mapping"]["tau"]]].copy()
        preview_df.columns = ["time", "tau"]
        plot_signal_panel(preview_df.head(min(2000, len(preview_df))), "time", ["tau"], "Utah FORGE primary signal preview", figsize=(12, 4))
    elif len(numeric_cols) >= 2:
        preview_df = raw_df[numeric_cols[:2]].head(min(2000, len(raw_df))).copy()
        preview_df.insert(0, "sample", range(len(preview_df)))
        plot_signal_panel(preview_df, "sample", numeric_cols[:2], "Utah FORGE numeric preview")
    else:
        print("No numeric preview available for plotting.")
else:
    print("Raw file missing. See dataset_summary.json for the exact file placement instructions.")
